# SETTING UP ACCESS TO GITHUB (PRESERVED STATE) FOR EVERY NEW RUNTIME

In [ ]:
from google.colab import userdata
import os

git_token = userdata.get('GIT_TOKEN')
git_username = "TalhaShoyo10"
repo_name = "CS-6304_PA0_28100131"
repo_url = f"https://{git_token}@github.com/{git_username}/{repo_name}.git"



if not os.path.exists(repo_name):
  !git clone {repo_url}
  print("Repo did not exist in local file system, cloned from github to initiate work on task.")
else:
  !git -C {repo_name} pull
  print("Repo already existed in local file system, pulled from github to catch up on any remote changes.")

Imports and Setup

In [ ]:
!pip install -q ftfy regex tqdm umap-learn
!pip install -q git+https://github.com/openai/CLIP.git

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import json
import os
from scipy.linalg import orthogonal_procrustes
from sklearn.manifold import TSNE
import clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CLIP TASKS

Loading CLIP and the STL-10 Dataset

In [ ]:
#Loading OpenAI's CLIP model (ViT-B/32 image encoder) along with its native preprocessing transform
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

#STL-10 test split is used for zero-shot evaluation (the model is never trained on this data)
test_dataset = torchvision.datasets.STL10(root="./data", split="test", download=True, transform=clip_preprocess)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

stl10_classes = test_dataset.classes
print(f"Number of test examples: {len(test_dataset)}")
print(f"Classes: {stl10_classes}")

# Zero-Shot Classification on STL-10

Building the Three Prompting Strategies

In [ ]:
#(i) Plain labels
plain_prompts = [c for c in stl10_classes]

#(ii) Prompted text ("a photo of a {class}")
photo_prompts = [f"a photo of a {c}" for c in stl10_classes]

#(iii) A more descriptive variant
descriptive_prompts = [f"a high quality photo of a {c}, a common object" for c in stl10_classes]

prompt_sets = {
    "plain_labels": plain_prompts,
    "a_photo_of_a": photo_prompts,
    "descriptive": descriptive_prompts,
}

for name, prompts in prompt_sets.items():
    print(f"{name}: {prompts[:3]} ...")

Encoding Text Prompts and Running Zero-Shot Evaluation

In [ ]:
def encode_text_prompts(prompts, clip_model, device):
    tokens = clip.tokenize(prompts).to(device)
    with torch.no_grad():
        text_features = clip_model.encode_text(tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features


def zero_shot_evaluate(loader, text_features, clip_model, device):
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            image_features = clip_model.encode_image(images)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            predicted = similarity.argmax(dim=-1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total


zero_shot_results = {}
for name, prompts in prompt_sets.items():
    text_features = encode_text_prompts(prompts, clip_model, device)
    accuracy = zero_shot_evaluate(test_loader, text_features, clip_model, device)
    zero_shot_results[name] = accuracy
    print(f"Prompting strategy: {name} -> Accuracy: {accuracy:.4f}")

Comparing the Prompting Strategies

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(zero_shot_results.keys(), zero_shot_results.values(), color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylabel("Zero-Shot Accuracy")
plt.title("STL-10 Zero-Shot Classification by Prompting Strategy")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task3_clip/zero_shot_accuracy_comparison.png", dpi=150)
plt.show()

# Exploring the Modality Gap

Extracting Image and Text Embeddings for a Sample of STL-10

In [ ]:
NUM_SAMPLES = 100

sample_loader = DataLoader(test_dataset, batch_size=NUM_SAMPLES, shuffle=True, num_workers=2)
sample_images, sample_labels = next(iter(sample_loader))
sample_images, sample_labels = sample_images.to(device), sample_labels.to(device)

with torch.no_grad():
    sample_image_features = clip_model.encode_image(sample_images)
    sample_image_features = sample_image_features / sample_image_features.norm(dim=-1, keepdim=True)

    #Encoding the corresponding class-name prompt for each sampled image, so every image has a paired text embedding
    sample_text_prompts = [f"a photo of a {stl10_classes[l]}" for l in sample_labels.cpu().tolist()]
    sample_tokens = clip.tokenize(sample_text_prompts).to(device)
    sample_text_features = clip_model.encode_text(sample_tokens)
    sample_text_features = sample_text_features / sample_text_features.norm(dim=-1, keepdim=True)

print(f"Image embeddings shape: {sample_image_features.shape}")
print(f"Text embeddings shape: {sample_text_features.shape}")

Projecting Both Modalities into 2D with t-SNE

In [ ]:
combined_embeddings = torch.cat([sample_image_features, sample_text_features], dim=0).cpu().numpy()
modality_labels = ["image"] * len(sample_image_features) + ["text"] * len(sample_text_features)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(combined_embeddings)

image_2d = embeddings_2d[:len(sample_image_features)]
text_2d = embeddings_2d[len(sample_image_features):]

plt.figure(figsize=(7, 6))
plt.scatter(image_2d[:, 0], image_2d[:, 1], label="image embeddings", alpha=0.6)
plt.scatter(text_2d[:, 0], text_2d[:, 1], label="text embeddings", alpha=0.6)
plt.legend()
plt.title("CLIP Modality Gap (t-SNE projection, pre-alignment)")
plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task3_clip/modality_gap_before_alignment.png", dpi=150)
plt.show()

#Quantifying separation: distance between the two modality centroids vs. average intra-modality spread
image_centroid = sample_image_features.mean(dim=0)
text_centroid = sample_text_features.mean(dim=0)
centroid_gap = (image_centroid - text_centroid).norm().item()
print(f"Distance between image and text centroids (normalized embeddings): {centroid_gap:.4f}")

Discussion points to fill in after reviewing the plot above:

- How separated are the two modalities in the projection?
- Does L2-normalization affect the apparent size of the modality gap (try comparing the centroid gap computed on raw vs. normalized embeddings)?
- Why does CLIP still perform well at zero-shot classification despite this gap (recall that classification only depends on relative cosine similarity ordering, not absolute overlap)?

# Bridging the Modality Gap

Learning an Orthogonal Procrustes Alignment

In [ ]:
#X: image embeddings, Y: text embeddings, both already paired sample-for-sample above
X = sample_image_features.cpu().numpy()
Y = sample_text_features.cpu().numpy()

#Solving for the orthogonal matrix R that minimizes || X @ R - Y ||_F
R, scale = orthogonal_procrustes(X, Y)

X_aligned = X @ R

print(f"Learned rotation matrix R with shape: {R.shape}")

Visualizing the Aligned Embeddings

In [ ]:
combined_aligned = np.concatenate([X_aligned, Y], axis=0)

tsne_aligned = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d_aligned = tsne_aligned.fit_transform(combined_aligned)

image_2d_aligned = embeddings_2d_aligned[:len(X_aligned)]
text_2d_aligned = embeddings_2d_aligned[len(X_aligned):]

plt.figure(figsize=(7, 6))
plt.scatter(image_2d_aligned[:, 0], image_2d_aligned[:, 1], label="image embeddings (aligned)", alpha=0.6)
plt.scatter(text_2d_aligned[:, 0], text_2d_aligned[:, 1], label="text embeddings", alpha=0.6)
plt.legend()
plt.title("CLIP Modality Gap (t-SNE projection, post-alignment)")
plt.tight_layout()
plt.savefig("CS-6304_PA0_28100131/results/task3_clip/modality_gap_after_alignment.png", dpi=150)
plt.show()

aligned_centroid_gap = np.linalg.norm(X_aligned.mean(axis=0) - Y.mean(axis=0))
print(f"Centroid gap before alignment: {centroid_gap:.4f}")
print(f"Centroid gap after alignment: {aligned_centroid_gap:.4f}")

Recomputing Zero-Shot Accuracy with Aligned Embeddings

In [ ]:
def zero_shot_evaluate_aligned(loader, text_features, clip_model, R_matrix, device):
    R_tensor = torch.from_numpy(R_matrix).float().to(device)
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            image_features = clip_model.encode_image(images)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            image_features = image_features.float() @ R_tensor

            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            predicted = similarity.argmax(dim=-1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return correct / total

#Using the best-performing prompting strategy from Subtask 1 as the text-side reference
best_prompt_name = max(zero_shot_results, key=zero_shot_results.get)
best_text_features = encode_text_prompts(prompt_sets[best_prompt_name], clip_model, device)

aligned_accuracy = zero_shot_evaluate_aligned(test_loader, best_text_features, clip_model, R, device)

print(f"Baseline accuracy ({best_prompt_name}): {zero_shot_results[best_prompt_name]:.4f}")
print(f"Accuracy after Procrustes alignment: {aligned_accuracy:.4f}")

Saving Results

In [ ]:
clip_results = {
    "zero_shot_accuracy_by_prompt": zero_shot_results,
    "centroid_gap_before_alignment": centroid_gap,
    "centroid_gap_after_alignment": float(aligned_centroid_gap),
    "best_prompting_strategy": best_prompt_name,
    "accuracy_before_alignment": zero_shot_results[best_prompt_name],
    "accuracy_after_alignment": aligned_accuracy,
    "num_samples_for_modality_gap": NUM_SAMPLES,
}

with open("CS-6304_PA0_28100131/results/task3_clip/clip_results.json", "w") as f:
    json.dump(clip_results, f, indent=2)

print("Results saved successfully !!")

Checking for Saved Files

In [ ]:
print(os.listdir("CS-6304_PA0_28100131/results/task3_clip"))

Git Configuration

In [ ]:
!git config --global user.email "thebenbat5@gmail.com"
!git config --global user.name "TalhaShoyo10"

Commiting work to Github

In [ ]:
commit_message = "Task 3 - CLIP zero-shot classification and modality gap analysis"
!git -C {repo_name} add -A
!git -C {repo_name} commit -m "{commit_message}"
!git -C {repo_name} push https://{git_token}@github.com/{git_username}/{repo_name}.git main